# Sandbox Execution

Run arbitrary Python scripts inside a Klavis Local Sandbox:
1. **Acquire** a sandbox
2. **Upload** a `tar.gz` archive containing `main.py` (and optionally `requirements.txt` + `packages.txt`)
3. **Execute** the script (background — poll logs to track progress)
4. **Export** artifacts produced by the execution
5. **Release** the sandbox

The sample script below writes output files to both the **execution directory** (CWD) and the **agent workspace** (`AGENT_WORKSPACE`), but this is not required — your script can do whatever it needs. Files in CWD are exportable via `/export-artifacts` API; files in `AGENT_WORKSPACE` persist across executions and are accessible by MCP servers.

## 1. Setup

In [ ]:
import asyncio
import io
import os
import tarfile
import time

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = "https://api.klavis.ai"
HEADERS = {"Authorization": f"Bearer {os.getenv('KLAVIS_API_KEY')}"}

## 2. Acquire a Sandbox

In [ ]:
async with httpx.AsyncClient() as client:
    resp = await client.post(
        f"{BASE_URL}/local-sandbox",
        headers=HEADERS,
        json={"server_names": ["filesystem", "terminal"]},
    )

acquire_data = resp.json()
SANDBOX_ID = acquire_data["local_sandbox_id"]
print(f"Sandbox ID: {SANDBOX_ID}")

## 3. Build a `tar.gz` Archive

The archive must contain a `main.py` (entry point) and optionally:
- `requirements.txt` — pip dependencies
- `packages.txt` — system (apt) packages to install

The script below uses `requests` + `psutil` (pip) and `curl` + `jq` (apt) to demonstrate both install paths.
It also writes output to **two locations**:
- **CWD** (execution dir) — `report.json`, exportable via `/export-artifacts`
- **`AGENT_WORKSPACE`** — `summary.txt`, persists across executions and is visible to MCP servers

In [ ]:
EXECUTION_NAME = "demo-run"

# -- main.py --
main_py = """\
import json
import os
import platform
import subprocess
from pathlib import Path

import requests  # pip dependency
import psutil     # pip dependency

print("=== Sandbox Execution Demo ===")
print(f"Python : {platform.python_version()}")
print(f"OS     : {platform.system()} {platform.release()}")
print(f"CPUs   : {psutil.cpu_count()}")
print(f"Memory : {psutil.virtual_memory().total // (1024**2)} MB")
print()

# Use 'curl' (system dep installed via packages.txt) to fetch a public API
print("--- curl (system dep) ---")
result = subprocess.run(
    ["curl", "-s", "https://httpbin.org/ip"],
    capture_output=True, text=True, timeout=15,
)
print(f"curl exit code: {result.returncode}")
curl_data = json.loads(result.stdout)
print(f"Public IP (via curl): {curl_data['origin']}")
print()

# Use 'requests' (pip dep) to fetch the same endpoint
print("--- requests (pip dep) ---")
resp = requests.get("https://httpbin.org/headers", timeout=15)
resp.raise_for_status()
print(f"Status: {resp.status_code}")
print(f"Host header seen by server: {resp.json()['headers'].get('Host')}")
print()

# Resolve the shared workspace directory
workspace = Path(os.environ.get("AGENT_WORKSPACE", "/data"))
workspace.mkdir(parents=True, exist_ok=True)

# List existing workspace files
if workspace.exists():
    files = sorted(str(p.relative_to(workspace)) for p in workspace.rglob("*") if p.is_file())
    print(f"Workspace files ({len(files)}):")
    for f in files[:20]:
        print(f"  {f}")
    print()

# Build a report dict
report = {
    "python_version": platform.python_version(),
    "platform": platform.platform(),
    "cpus": psutil.cpu_count(),
    "memory_mb": psutil.virtual_memory().total // (1024**2),
    "public_ip": curl_data["origin"],
}

# --- Write to CWD (execution dir) ---
# Files here are exportable via /export-artifacts
cwd_output = Path("report.json")
cwd_output.write_text(json.dumps(report, indent=2))
print(f"[CWD]       Wrote {cwd_output} ({cwd_output.stat().st_size} bytes)")

# --- Write to AGENT_WORKSPACE ---
# Files here persist across executions and are visible to MCP servers
ws_output = workspace / "summary.txt"
ws_output.write_text(
    f"Execution completed on {platform.node()}\\n"
    f"Python {platform.python_version()}, {psutil.cpu_count()} CPUs, "
    f"{psutil.virtual_memory().total // (1024**2)} MB RAM\\n"
    f"Public IP: {curl_data['origin']}\\n"
)
print(f"[WORKSPACE] Wrote {ws_output} ({ws_output.stat().st_size} bytes)")
"""

# -- requirements.txt (pip dependencies) --
requirements_txt = "requests\npsutil\n"

# -- packages.txt (system/apt dependencies) --
packages_txt = "curl\njq\n"

# Pack into tar.gz
buf = io.BytesIO()
with tarfile.open(fileobj=buf, mode="w:gz") as tar:
    for name, content in [
        ("main.py", main_py),
        ("requirements.txt", requirements_txt),
        ("packages.txt", packages_txt),
    ]:
        data = content.encode()
        info = tarfile.TarInfo(name=name)
        info.size = len(data)
        tar.addfile(info, io.BytesIO(data))
buf.seek(0)
print(f"Archive ready: {buf.getbuffer().nbytes} bytes, 3 files")

## 4. Upload → Execute

1. `POST /execution-upload-url` — get a signed GCS upload URL
2. `PUT` the archive to that URL
3. `POST /execute` — trigger execution inside the sandbox

In [ ]:
# Step 1: Get upload URL
async with httpx.AsyncClient() as client:
    resp = await client.post(
        f"{BASE_URL}/local-sandbox/{SANDBOX_ID}/execution-upload-url",
        headers=HEADERS,
        json={"execution_name": EXECUTION_NAME},
    )
    resp.raise_for_status()
upload_url = resp.json()["upload_url"]
print(f"Upload URL obtained (expires in {resp.json()['expires_in_minutes']} min)")

# Step 2: Upload the archive
async with httpx.AsyncClient(timeout=60.0) as client:
    resp = await client.put(
        upload_url,
        headers={"Content-Type": "application/gzip"},
        content=buf.getvalue(),
    )
    resp.raise_for_status()
print(f"Upload complete (HTTP {resp.status_code})")

# Step 3: Execute
async with httpx.AsyncClient(timeout=60.0) as client:
    resp = await client.post(
        f"{BASE_URL}/local-sandbox/{SANDBOX_ID}/execute",
        headers=HEADERS,
        json={"execution_name": EXECUTION_NAME},
    )
    resp.raise_for_status()
print(resp.json())

## 5. Poll Status & Logs

Execution runs in the background. Poll until `completed` or `failed`.

In [ ]:
cursor = 0

async with httpx.AsyncClient(timeout=30.0) as client:
    while True:
        # Poll logs (includes status + new output since cursor)
        resp = await client.get(
            f"{BASE_URL}/local-sandbox/{SANDBOX_ID}/execution-logs",
            headers=HEADERS,
            params={"execution_name": EXECUTION_NAME, "cursor": cursor},
        )
        resp.raise_for_status()
        data = resp.json()

        if data["logs"]:
            print(data["logs"], end="")
        cursor = data["cursor"]

        if data["status"] in ("completed", "failed"):
            print(f"\n--- Execution {data['status']} ---")
            break

        await asyncio.sleep(2)

## 6. Check Status (standalone)

You can also check status without fetching logs.

In [ ]:
async with httpx.AsyncClient() as client:
    resp = await client.get(
        f"{BASE_URL}/local-sandbox/{SANDBOX_ID}/execution-status",
        headers=HEADERS,
        params={"execution_name": EXECUTION_NAME},
    )
    resp.raise_for_status()
print(resp.json())

## 7. Export Artifacts

Download files produced by the execution (e.g. `report.json`).

In [ ]:
# Request artifact export (server zips execution dir → GCS → returns download URL)
async with httpx.AsyncClient(timeout=120.0) as client:
    resp = await client.post(
        f"{BASE_URL}/local-sandbox/{SANDBOX_ID}/export-artifacts",
        headers=HEADERS,
        json={"execution_name": EXECUTION_NAME},
    )
    resp.raise_for_status()
export_data = resp.json()
print(f"Download URL obtained (expires in {export_data['expires_in_minutes']} min)")

# Download and inspect the artifact archive
async with httpx.AsyncClient(timeout=120.0) as client:
    resp = await client.get(export_data["download_url"])
    resp.raise_for_status()

with tarfile.open(fileobj=io.BytesIO(resp.content), mode="r:gz") as tar:
    print("Exported artifacts:")
    for m in tar.getmembers():
        print(f"  {m.name} ({m.size} bytes)")
        if m.name.endswith(".json") and m.isfile():
            f = tar.extractfile(m)
            if f:
                print(f.read().decode())

## 8. Release the Sandbox

In [ ]:
async with httpx.AsyncClient() as client:
    resp = await client.delete(
        f"{BASE_URL}/local-sandbox/{SANDBOX_ID}",
        headers=HEADERS,
    )
    resp.raise_for_status()
print(resp.json())